# Практическая работа 2. Работа с LLM через API
## Демонстрационный ноутбук: базовое промптирование и извлечение данных

- **Студент:** Григорьев Михаил Романович
- **Группа:** ИСТ.04 (15.27Д-ИСТ04/26б)
- **Вариант:** 06 (`rental_06.csv`)
- **Ветка:** `env-feature`


### 1. Настройка окружения и импорт компонентов LangChain


In [1]:
import os
from dotenv import load_dotenv

# Загружаем переменные окружения из .env
load_dotenv()
GIGA_KEY = os.getenv("GIGA_KEY")

# Импорт основных компонентов LangChain
from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)
from langchain_core.output_parsers import StrOutputParser


### 2. Базовая инициализация модели GigaChat


In [2]:
# Настройка модели GigaChat с расширенными параметрами
llm = GigaChat(
    credentials=GIGA_KEY,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000
)

# Проверка подключения
response = llm.invoke("Привет! Как дела?")
print(response.content)


Привет! Всё отлично, готов общаться и помогать тебе с любыми вопросами. Расскажи, что интересного у тебя происходит?


### 3. Системное и пользовательское промптирование (ChatPromptTemplate)


In [3]:
# Шаблон системного сообщения
system_prompt = SystemMessagePromptTemplate.from_template(
    "Ты — эксперт по анализу текстовых заявок на аренду жилья."
)

# Шаблон пользовательского сообщения
user_prompt = HumanMessagePromptTemplate.from_template(
    "Заявка: {text}\n\nИзвлеки количество человек и верни только число."
)

# Собираем чат-шаблон
chat_prompt = ChatPromptTemplate.from_messages([system_prompt, user_prompt])

# Создание цепочки
chain = chat_prompt | llm | StrOutputParser()

# Тестовый вызов цепочки
result = chain.invoke({"text": "Семья из трех человек снимает квартиру"})
print(f"Результат: {result}") # ожидаем '3'


Результат: 3


### 4. Базовое промптирование через PromptTemplate


In [4]:
# Простой промпт для извлечения количества людей
basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.

Текст заявки: {text}

Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.

Количество человек:"""
)

# Создание цепочки с базовым промптом
basic_chain = basic_prompt | llm | StrOutputParser()

# Первичное тестирование на простых примерах
test_texts = [
    "Ищу квартиру для семьи из четырех человек",
    "Нужна студия для одного",
    "Семья с двумя детьми ищет жилье"
]

for text in test_texts:
    res = basic_chain.invoke({"text": text})
    print(f"Текст: {text}")
    print(f"Результат: {res}")
    print("---")


Текст: Ищу квартиру для семьи из четырех человек
Результат: 4
---
Текст: Нужна студия для одного
Результат: 1
---
Текст: Семья с двумя детьми ищет жилье
Результат: 4
---


### 5. Индивидуальная работа: тестирование на 15 заявках варианта 06
Загружаем заявки варианта 06 (`rental_06.csv`), помещаем их в `test_texts` и запускаем цепочку базового промптирования.


In [5]:
import pandas as pd
import time

# Загружаем датасет варианта 06
df_demo = pd.read_csv("rental_06.csv", sep=";")

test_texts = df_demo["text"].tolist()

print(f"Загружено заявок: {len(test_texts)}\n")

demo_results = []
for idx, text in enumerate(test_texts, start=1):
    time.sleep(0.3)
    try:
        ans = basic_chain.invoke({"text": text}).strip()
        demo_results.append(ans)
    except Exception as e:
        ans = f"ERROR: {e}"
        demo_results.append(ans)
    print(f"{idx}. Текст: {text[:65]}...")
    print(f"   Результат LLM: {ans}")
    print("---")


Загружено заявок: 15

1. Текст: ищем жилье в центре недалеко от моря с 23.07-03.08 - нужен 1 двухмес...
   Результат LLM: 5
---
2. Текст: Сниму недорогое жилье в лазаревском на 6 чел 3 взр и 3 дет 30.07-09....
   Результат LLM: 6
---
3. Текст: Сниму не дорогое жилье 1 взрослый и 2 ребенка ( 9 лет и 5 лет). П...
   Результат LLM: 3
---
4. Текст: Ищем жильё в Лазаревском,желательно улица Павлова,не далеко от мо...
   Результат LLM: 4
---
5. Текст: Интересует жилье на одного человека с 08.07. до 15.07. Не далеко от...
   Результат LLM: 1
---
6. Текст: Здравствуйте! Интересует жилье с 11 по 22 августа. Один взрослый ...
   Результат LLM: 2
---
7. Текст: Здравствуйте, интересует жильё с 7.09 по 14.09. На двоих взрослых. ...
   Результат LLM: 2
---
8. Текст: Интересует жилье в Сочи у пляжа "Ривьера" (максимально близко к мо...
   Результат LLM: 3
---
9. Текст: Лазаревское, Солоники - бюджетное жилье на 3 суток 4 взрослых с 5...
   Результат LLM: 4
---
10. Текст: Ищем жилье в Лазаревском с 12